# 🔬 KLA Hackathon — NAFNet-SR Image Restoration (Kaggle Edition)
**12-Hour Continuous GPU Execution — No Disconnections!**

### Instructions:
1. In the right sidebar under **Notebook Options**:
   - Set **Accelerator** → `GPU T4 x2` or `GPU P100`
   - Set **Internet** → `On`
2. Upload your `train.zip` and `test_noisyLR.zip` via **+ Add Input** → **Upload a dataset** (or upload to Kaggle Working directory)
3. Click **Run All**!

In [ ]:
# CELL 1: Check Kaggle GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2: Clone repository & install requirements
import os
%cd /kaggle/working
!rm -rf /kaggle/working/kla_restoration
!git clone https://github.com/norriy0u/kla-image-restoration.git /kaggle/working/kla_restoration
%cd /kaggle/working/kla_restoration
!pip install -r requirements.txt -q
print('✓ Setup complete!')

In [ ]:
# CELL 3: Auto-detect dataset in /kaggle/input or extract zips
import os, glob, shutil

LOCAL_DATA = '/kaggle/working/kla_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# Search /kaggle/input for existing extracted datasets or zips
print('Scanning /kaggle/input for dataset...')
input_npys = []
for root, _, files in os.walk('/kaggle/input'):
    npys = [f for f in files if f.endswith('.npy')]
    if len(npys) > 50:
        input_npys.append((len(npys), root))

input_npys.sort(reverse=True)
print('Found dataset directories in /kaggle/input:')
for count, path in input_npys:
    print(f'  {count:4d} files  →  {path}')

def pick(dirs, *keywords, exclude=()):
    for _, p in dirs:
        name = p.lower()
        if all(k.lower() in name for k in keywords) and not any(e.lower() in name for e in exclude):
            return p
    return None

GT_DIR   = pick(input_npys, 'gt',    exclude=('noisy','test'))
LR_DIR   = pick(input_npys, 'noisy', exclude=('test',))
TEST_DIR = pick(input_npys, 'test')

print(f'\nGT_DIR   = {GT_DIR}')
print(f'LR_DIR   = {LR_DIR}')
print(f'TEST_DIR = {TEST_DIR}')

gt_count = len(glob.glob(f'{GT_DIR}/*.npy')) if GT_DIR else 0
lr_count = len(glob.glob(f'{LR_DIR}/*.npy')) if LR_DIR else 0
print(f'\nGT files: {gt_count} | LR files: {lr_count}')
assert gt_count > 0 and lr_count > 0, 'Could not find GT/LR files in /kaggle/input! Please upload dataset to Kaggle.'
print('✓ Dataset ready!')

In [ ]:
# CELL 4: Quick Data Inspection
import numpy as np, matplotlib.pyplot as plt, glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, idx in enumerate([0, len(gt_files)//3, 2*len(gt_files)//3, len(gt_files)-1]):
    gt = np.load(gt_files[idx])
    lr = np.load(lr_files[idx])
    axes[0][col].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0][col].set_title(f'GT #{idx} {gt.shape}', fontsize=9)
    axes[0][col].axis('off')
    axes[1][col].imshow(np.clip(lr,0,1), cmap='gray', vmin=0, vmax=1)
    axes[1][col].set_title(f'NoisyLR #{idx} {lr.shape}', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Sample Training Pairs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_pairs.png', dpi=150)
plt.show()

In [ ]:
# CELL 5: TRAIN (Runs up to 12 hours non-stop!)
%cd /kaggle/working/kla_restoration
OUT_WEIGHTS = '/kaggle/working/weights'
os.makedirs(OUT_WEIGHTS, exist_ok=True)

!PYTHONUNBUFFERED=1 python train.py \
    --gt_dir {GT_DIR} \
    --lr_dir {LR_DIR} \
    --epochs 200 \
    --batch_size 16 \
    --num_workers 4 \
    --val_fraction 0.1 \
    --patch_size_gt 256 \
    --model_variant base \
    --weights_dir {OUT_WEIGHTS} \
    --log_dir /kaggle/working/logs \
    --no_amp

In [ ]:
# CELL 6: Evaluation Metrics
import os, json
os.makedirs('/kaggle/working/val_outputs', exist_ok=True)

!python /kaggle/working/kla_restoration/evaluate.py \
    --input_dir {LR_DIR} \
    --output_dir /kaggle/working/val_outputs \
    --gt_dir {GT_DIR} \
    --weights /kaggle/working/weights/best_model.pt \
    --batch_size 16

with open('/kaggle/working/val_outputs/metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# CELL 7: Test Set Inference
import os, json
if TEST_DIR:
    os.makedirs('/kaggle/working/test_outputs', exist_ok=True)
    !python /kaggle/working/kla_restoration/evaluate.py \
        --input_dir {TEST_DIR} \
        --output_dir /kaggle/working/test_outputs \
        --weights /kaggle/working/weights/best_model.pt \
        --batch_size 16
    with open('/kaggle/working/test_outputs/metrics.json') as f:
        print(json.dumps(json.load(f), indent=2))